This is an example of generating MODIS-LAI h5 files to drive ATS 2D transect simulations.

- modified based on `get_MODIS-LAI.ipynb` in watershed-workflow
- Input
    - `data-processed/{site_name}/m2_coords_{site_name}.mat`
    - `notebooks/MODIS_raw` data downloaded from APPEEARS website
- Output
    - `data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_20021001_20250125.h5`: raw MODIS
    - `data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_2013-1-1_2024-1-1_smoothed.h5`: smoothed MODIS for transient simulation
    - `data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_typical10yr_2013_2023.h5`: typical year MODIS for spinup simulation
    - `data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_merged.h5`: merged spinup and transient
    - ~~`data-processed/{site_name}/{site_name}_MODIS_LAI_20021001_20250125.h5`: raw MODIS for hillslope~~
    - ~~`data-processed/{site_name}/{site_name}_MODIS_LAI_2013-1-1_2024-1-1_smoothed.h5`: smoothed MODIS for specified start_year to end_year~~
    - ~~`data-processed/{site_name}/{site_name}_MODIS_LAI_typical10yr_2013_2023.h5`: typical year MODIS for spinup~~

**File History**

update 2026/2/2
- revise hillslope LAI calculation
- previously,
    1. obtain crosswalk of NLCD labels and MODIS labels
    2. for each MODIS label, calculate the 'watershed' averaged LAI
    3. link domain grid to NLCD label, then to an averaged LAI-time of corresponding MODIS label
- now, improve the representation of local LAI changes
    1. obtain crosswalk of NLCD and MODIS labels
    2. hillslope -> find NLCD grids
    3. for each hillslope NLCD grids, find closest MODIS grid with corresponding MODIS label (from crosswalk)
    4. calculate averaged MODIS LAI-time, if multiple MODIS grids were found with the same MODIS label
- [note] revise the [4.surface properties] part of 1a-main_workflow_Naches.ats1.5.ipynb

update 2026/1/21
- add merged MODIS-LAI of spinup and transient for easier restart

update 2025/10/13
- update `config.json`. Mainly revise the model run pipeline.

update 2025/8/32
- add `config.json`

In [ ]:
%load_ext autoreload
%autoreload 2

# Parameters and data sources

In [ ]:
# Parameters cell -- schema-v2 forcing timeline configuration
from config_utils import (load_config, phase_dates, phase_label, noleap_day_of_year,
                          phase_forcing_dir, full_timeline_forcing_dir)
config = load_config('config.json')
case = config['case']; watershed_name = case['watershed_name']; hucs = case['hucs']; site_name = case['site_name']; meshsize_nx = case['meshsize_nx']
spinup_dates = phase_dates(config, 'spinup'); prefire_dates = phase_dates(config, 'prefire_transient')
postfire_dates = phase_dates(config, 'postfire_transient') if 'postfire_transient' in config else []
spinup_label = phase_label(config, 'spinup'); prefire_label = phase_label(config, 'prefire_transient')
postfire_label = phase_label(config, 'postfire_transient') if postfire_dates else None
nyears_steadystate_spinup = config['spinup']['steady_state_years']; nyears_cyclic_spinup = config['spinup']['cyclic_years']
forcing_spinup_dir = phase_forcing_dir(config, 'spinup'); forcing_prefire_dir = phase_forcing_dir(config, 'prefire_transient')
forcing_postfire_dir = phase_forcing_dir(config, 'postfire_transient') if postfire_dates else None
forcing_full_dir = full_timeline_forcing_dir(config)
for _d in (forcing_spinup_dir, forcing_prefire_dir, forcing_postfire_dir, forcing_full_dir):
    if _d is not None: _d.mkdir(parents=True, exist_ok=True)


In [ ]:
outputs={}

In [ ]:
import os, sys
import xarray as xr # rioxarray required
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py as h5
import geopandas as gpd
from shapely.geometry import mapping
import netCDF4 as nc
from datetime import date, datetime, timedelta
import calendar
import shutil

import geopandas as gpd
from scipy.io import loadmat
import shapely
from shapely.geometry import Point, LineString, Polygon, box, mapping

import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.ui
import watershed_workflow.colors
import watershed_workflow.condition
import watershed_workflow.mesh
import watershed_workflow.split_hucs
import watershed_workflow.soil_properties
import watershed_workflow.daymet
import watershed_workflow.utils
import watershed_workflow.regions
import watershed_workflow.land_cover_properties

import scipy
import pyproj

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = (9,5)
pd.options.display.max_columns = None
pd.options.display.max_rows = 20

In [ ]:
# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs_daymet = watershed_workflow.crs.daymet_crs()
crs_latlon = watershed_workflow.crs.latlon_crs() # essentially epsg(4269)
# note: epsg(4269) i.e. NAD83 vs epsg(4326) i.e. WGS84
# - https://gis.stackexchange.com/questions/170839/is-re-projection-needed-from-srid-4326-wgs-84-to-srid-4269-nad-83

# alternative
#proj_daymet = "+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +datum=WGS84" # daymet crs
#proj_wgs84  = "epsg:4326" # latlon
#crs_daymet  = watershed_workflow.crs.from_string(proj_daymet)
#crs_wgs84   = watershed_workflow.crs.from_string(proj_wgs84)

# prepare watershed shape and hillslope shape

In [ ]:
# load from watershed shp
#watershed_name = 'OakCreek' # name the domain, used in filenames, etc
fname_watershed_shp = f'../data-processed/{watershed_name}/{watershed_name}_bounds.shp'
watershed_shape = gpd.read_file(fname_watershed_shp)

In [ ]:
# ## Import shapefile to provide the coordinates - not used anymore?
# def get_bounds(fname_watershed_shp):
#     """get the min,max bounds for lat,lon for a given watershed."""
#     watershed_shape = gpd.read_file(fname_watershed_shp)

#     bounds = watershed_shape.bounds
#     # bounds

#     lonl = watershed_shape.bounds['minx'].values[0]
#     lonr = watershed_shape.bounds['maxx'].values[0]
#     latb = watershed_shape.bounds['miny'].values[0]
#     latt = watershed_shape.bounds['maxy'].values[0]    
    
#     return lonl,lonr,latb,latt

# lonl,lonr,latb,latt = get_bounds(fname_watershed_shp)

# lonl,lonr,latb,latt

In [ ]:
# load hillslope geometry from mat file generated in "1-full_workflow_OakCreek.ipynb"
#site_name = 'NF01'
meshsize_nx = 100

m2_mat_filename =  f'../data-processed/{site_name}/m2_coords_{site_name}.mat'
loaded_data = loadmat(m2_mat_filename)
#dzs_soil  = loaded_data['dzs_soil'].flatten()
#dzs_geo   = loaded_data['dzs_geo'].flatten()
#m2_coords = loaded_data['m2_coords']
loaded_gdf_dict = loaded_data['gdf_data']
gdf_reloaded = pd.DataFrame({
    'lon': loaded_gdf_dict['lon'][0, 0].flatten(),
    'lat': loaded_gdf_dict['lat'][0, 0].flatten(),
    'h_distance': loaded_gdf_dict['h_distance'][0, 0].flatten(),
    'elevation': loaded_gdf_dict['elevation'][0, 0].flatten()
})
geometry = [Point(xy) for xy in zip(gdf_reloaded['lon'], gdf_reloaded['lat'])]
hillslope_gdf = gpd.GeoDataFrame(gdf_reloaded, geometry=geometry)

# create hillslope polygon and shape object
xsec_plg = Polygon([hillslope_gdf.geometry[i] for i in range(hillslope_gdf.shape[0])])
xsec_plg_dict = {"type": "Feature", "id":0, "properties":{}, "geometry": mapping(xsec_plg)}
xsec_plg_dict_shply = watershed_workflow.utils.create_shply(xsec_plg_dict)

# convert to latlon crs, used in some plots
reproj_xsec_plg = watershed_workflow.warp.shape(xsec_plg_dict, crs_daymet, crs_latlon)
reproj_xsec_plg_shply = watershed_workflow.utils.create_shply(reproj_xsec_plg)

In [ ]:
hillslope_gdf

# Get MODIS-LAI for the watershed

MODIS product:

|Product|File name|Spatial resolution|Temporal resolution|Period|
|---|---|---|---|---|
|LAI|MCD15A3H.0.61_500m_aid0001.nc|500-m|4-day|2002-07-01 - present|
|Landcover|MCD12Q1.0.61_500m_aid0001.nc|500-m|yearly|2001-01-01 - present|
|ET|MOD16A2GF.0.61_500m_aid0001.nc|500-m|8-day|2000-01-01 - present|
|Snowcover|MOD10A2.0.61_500m_aid0001.nc|500-m|6-day?|2000-02-24 - present|

## MODIS-LAI data: load, subset, and plot

In [ ]:
#watershed_name = 'OakCreek'
data_raw_dir = f'./MODIS_raw/{watershed_name}/{watershed_name}-square'

fname_lai = data_raw_dir + '/MCD15A3H.061_500m_aid0001.nc'
fname_lulc = data_raw_dir + '/MCD12Q1.061_500m_aid0001.nc'
fname_et = data_raw_dir + '/MOD16A2GF.061_500m_aid0001.nc'
fname_snowcover = data_raw_dir + '/MOD10A2.061_500m_aid0001.nc'

In [ ]:
dset = xr.open_dataset(fname_lai)
dset

In [ ]:
data = dset.Lai_500m
data.shape

In [ ]:
np.nanmin(data.values), np.nanmax(data.values)

In [ ]:
np.nanmean(data.values), np.nanmedian(data.values)

In [ ]:
plt.hist(data.values.flatten()[data.values.flatten()<1000], 100)
plt.show()

In [ ]:
# The upper LAI validity screen of 10 was selected from the histogram above.
# It removes clearly invalid/extreme source values; it is not the temporal
# despike threshold applied later to the hillslope LAI time series.
mask_data = data.where((data < 10) & (data >= 0))
mask_data.min(), mask_data.max()

In [ ]:
#mask_data.isel(time=-1).plot()

In [ ]:
mask_data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
mask_data.rio.write_crs("epsg:4326", inplace=True)
clipped_data = mask_data.rio.clip(watershed_shape.geometry.apply(mapping), watershed_shape.crs, drop = True)

In [ ]:
LAI_data = clipped_data
LAI_data.shape

In [ ]:
LAI_data.isel(time=slice(1700, 1790, 8)).plot(x="lon", y='lat', col="time", col_wrap=4, robust=True, levels=np.linspace(0,5,51), cmap='Spectral_r')

In [ ]:
# Schneider Springs Fire: 2021-8-4 evening to ??

make_many_figs_and_gif = False
make_single_fig = True

# if make_many_figs_and_gif:
    
#     desktop_dir = os.environ.get('DESKTOPDIR')
#     temp_imgs_dir = desktop_dir + '/LAI_temp_imgs/'

#     try:
#         shutil.rmtree(temp_img_dir)
#     except OSError:
#         pass
#     os.mkdir(temp_img_dir)
    
#     for i in np.arange(int(LAI_data.shape[0]/10)):
#         fig, ax = plt.subplots(1, 1, figsize=(8,4))
#         LAI_data.isel(time=i).plot(ax=ax, levels=np.linspace(0,5,51), cmap='Spectral_r', extend='max')
#         # watershed_shape.boundary.plot(ax=ax, color='r')
#         ax.set_aspect('equal')
#         plt.tight_layout()
#         plt.savefig(temp_img_dir + str(i).zfill(4)+'.jpg')
#         plt.close()
#         if i % 100 == 0:
#             print(i, end=' ')
#     import imageio
#     from pygifsicle import optimize
#     images, image_file_names = [], []
#     for file_name in os.listdir(temp_img_dir):
#         if file_name.endswith('.jpg'):
#             image_file_names.append(file_name)       
#     # sorted_files = sorted(image_file_names, key=lambda y: int(y.split('_')[1]))
#     for i in range(len(image_file_names)):       
#         filetemp_img_dir = os.path.join(temp_img_dir, image_file_names[i])
#         images.append(imageio.imread(filetemp_img_dir))
#     imageio.mimsave(desktop_dir + f'{name}_LAI.gif', images, 'GIF', loop=1, fps=30)
#     optimize(desktop_dir + f'{name}_LAI.gif')
    
if make_single_fig:
    fig, axs = plt.subplots(1, 2, figsize=(10,4))
    ax1, ax2 = axs[0], axs[1]
    LAI_data.isel(time=1751).plot(ax=ax1, levels=np.linspace(0,5,51), cmap='Spectral_r', extend='max')
    LAI_data.isel(time=1760).plot(ax=ax2, levels=np.linspace(0,5,51), cmap='Spectral_r', extend='max')
    plt.tight_layout()
    plt.show()


In [ ]:
# Schneider Springs Fire: 2021-8-4 evening to ??
print("pre-fire time: " + LAI_data.time[1751].dt.strftime('%Y-%m-%d').item())
print("post-fire time: " + LAI_data.time[1760].dt.strftime('%Y-%m-%d').item())

fig, ax = plt.subplots(1, 1, figsize=(5,4))
dif = LAI_data.isel(time=1751) - LAI_data.isel(time=1760)
dif.plot(ax=ax, levels=np.linspace(-2.5,2.5,51), cmap='coolwarm', extend='both')
plt.title('positive: pre-fire > post-fire\nnegative: pre-fire < post-fire')
plt.tight_layout()
plt.show()

## MODIS-LULC data: load, subset, and plot

- LULC Land Use and Land Cover

In [ ]:
dset = xr.open_dataset(fname_lulc)
dset

In [ ]:
# counting LULC types
data = dset.LC_Type1
#ids, counts = np.unique(data.values[~np.isnan(data.values)], return_counts=True)

mask_data = data
mask_data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
mask_data.rio.write_crs("epsg:4326", inplace=True)
landcover_data_modis_masked = mask_data.rio.clip(watershed_shape.geometry.apply(mapping), watershed_shape.crs, drop = True)

In [ ]:
# MODIS LULC labels
# Colors are based on NLCD LULC colors
lc_type1_colors = {
        -1:  ('Unclassified', (0.00000000000,  0.00000000000,  0.00000000000)),
        0: ('Open Water', (0.27843137255,  0.41960784314,  0.62745098039)),
        1: ('Evergreen Needleleaf Forests', (0.10980392157,  0.38823529412,  0.18823529412)),
        2: ('Evergreen Broadleaf Forests', (0.10980392157,  0.38823529412,  0.18823529412)),
        3: ('Deciduous Needleleaf Forests', (0.40784313726,  0.66666666667,  0.38823529412)),
        4: ('Deciduous Broadleaf Forests', (0.40784313726,  0.66666666667,  0.38823529412)),
        5: ('Mixed Forests', (0.70980392157,  0.78823529412,  0.55686274510)),
        6: ('Closed Shrublands', (0.80000000000,  0.72941176471,  0.48627450980)),
        7: ('Open Shrublands', (0.80000000000,  0.72941176471,  0.48627450980)),
        8: ('Woody Savannas', (0.60980392157,  0.68823529412,  0.55686274510)),
        9: ('Savannas', (0.70980392157,  0.78823529412,  0.55686274510)),
        10: ('Grasslands', (0.88627450980,  0.88627450980,  0.75686274510)),
        11: ('Permanent Wetlands', (0.43921568628,  0.63921568628,  0.72941176471)),
        12: ('Croplands', (0.66666666667,  0.43921568628,  0.15686274510)),
        13: ('Urban and Built up lands', (0.86666666667,  0.78823529412,  0.78823529412)),
        14: ('Cropland Natural Vegetation Mosaics', (0.66666666667,  0.43921568628,  0.15686274510)),
        15: ('Permanent Snow and Ice', (0.81960784314,  0.86666666667,  0.97647058824)),
        16: ('Barren Land', (0.69803921569,  0.67843137255,  0.63921568628)),
        17: ('Water Bodies', (0.27843137255,  0.41960784314,  0.62745098039)),
    } 

In [ ]:
lc_colors = lc_type1_colors

In [ ]:
watershed_ids, watershed_counts = np.unique(landcover_data_modis_masked.values[~np.isnan(landcover_data_modis_masked.values)], return_counts=True)

watershed_colors = [lc_colors[i][1] for i in watershed_ids]
watershed_labels = [lc_colors[i][0] for i in watershed_ids]

watershed_ids, watershed_counts, watershed_colors, watershed_labels

In [ ]:
watershed_ids.max()

In [ ]:
labelsp1 = [lc_colors[i][0] for i in lc_colors]

counts, bins = np.histogram(landcover_data_modis_masked, range=[-1,17], bins=18)
plt.hist(bins[:-1], bins=18, range=[-1,17], weights=counts)
plt.xlabel("MODIS LULC")
plt.ylabel("Counts")
plt.title("Histogram of number of pixels per LULC types")
plt.xticks(bins+0.5,labelsp1,rotation=90)
plt.show()

In [ ]:
# LULC data plot - spatial distribution at a time slice

# [Yi] add an id to ensure plot(levels=watershed_ids, colors = watershed_colors, ax=ax, add_colorbar = False) display correctly
# watershed_ids_ext4plot = [0] + watershed_ids.tolist() + [watershed_ids.max()+1]
# watershed_colors_ext4plot = ['grey'] + watershed_colors + ['k']
# watershed_labels_ext4plot = ['None'] + watershed_labels + ['']
watershed_ids_ext4plot = watershed_ids.tolist() + [watershed_ids.max()+1]
watershed_colors_ext4plot = watershed_colors + ['k']
watershed_labels_ext4plot = watershed_labels + ['']

fig, ax = plt.subplots(1,1, figsize=(8,4))
g = landcover_data_modis_masked.isel(time = -1).plot(levels=watershed_ids_ext4plot, 
                                                     colors = watershed_colors_ext4plot, ax=ax, add_colorbar = False)

midpoints = 0.5 * (np.array(watershed_ids_ext4plot[:-1]) + np.array(watershed_ids_ext4plot[1:]))
cb = plt.colorbar(g)
cb.set_ticks(midpoints)
cb.ax.tick_params(size=0)
cb.set_ticklabels(watershed_labels_ext4plot[:-1])

watershed_shape.boundary.plot(ax=ax, color ='r')
ax.set_aspect('equal')

## Merge LAI and LULC to a dataframe

In [ ]:
# LULC data plot - temporal plot

ilandcover = landcover_data_modis_masked.sel(time="2023-01-01").values[0,:,:] # in watershed-workflow, by default it's using the latest year LULC.
print(ilandcover.shape)

# lc_ids = np.unique(landcover_data_modis_masked.values[~np.isnan(landcover_data_modis_masked.values)])
# print(lc_ids)
# lc_labels = [lc_colors[i][0] for i in lc_ids]
# print(lc_labels)
times = LAI_data.time.values
print(times)
print(len(times))

In [ ]:
# calculate mean LAI for each landcover type
LAI_lc = []

for itime in times:
#itime = times[0]
    iCC_LAI = LAI_data.sel(time=itime).values
    iLAI_lc = []
    for i,ilabel in zip(watershed_ids, watershed_labels):
        idx = np.where(ilandcover == i)
        coords = list(zip(idx[0], idx[1]))
        # choose mean of the LAI for each landcover type
        iLAI = np.nanmean(np.array([iCC_LAI[i] for i in coords]))
        iLAI_lc.append(iLAI)
    LAI_lc.append(iLAI_lc)
    
print(len(LAI_lc))
print(LAI_lc[0])

In [ ]:
# generate dataframe: LAI of each LC with time
LAI_df = pd.DataFrame(LAI_lc, columns=watershed_labels)
LAI_df['datetime'] = LAI_data.indexes['time'].to_datetimeindex()
# LAI_df = LAI_df.iloc[21:-29]
LAI_df = LAI_df.iloc[0:]
LAI_df.iloc[0,-1] = LAI_df.iloc[0,-1]+(LAI_df.iloc[1,-1]-LAI_df.iloc[0,-1])/4
LAI_df.set_index('datetime', inplace=True)
LAI_df['time [s]'] = (LAI_df.index - LAI_df.index[0]).total_seconds()
LAI_df

In [ ]:
LAI_df.plot()
plt.ylabel('LAI [-]')
plt.xlim(["2005-01-01", "2010-01-01"])
plt.ylim(-0.5,5)

### [optional] LAI varition plot

In [ ]:
# if want to see the spatial variation

# Dictionary to store spatial variation (per coordinate) for each watershed
spatial_variation = {label: {} for label in watershed_labels}

for itime in times:
    iCC_LAI = LAI_data.sel(time=itime).values
    for i, ilabel in zip(watershed_ids, watershed_labels):
        # Get all coordinates for the current landcover type
        idx = np.where(ilandcover == i)
        coords = list(zip(idx[0], idx[1]))
        
        # Iterate over each coordinate and store time series
        for coord in coords:
            if coord not in spatial_variation[ilabel]:
                spatial_variation[ilabel][coord] = []
            spatial_variation[ilabel][coord].append(iCC_LAI[coord])

# Plot spatial variations for each landcover type
times_datetime = LAI_data.indexes['time'].to_datetimeindex()

# for ilabel, coord_data in spatial_variation.items():
#     plt.figure(figsize=(10, 6))
#     for coord, values in coord_data.items():
#         plt.plot(times_datetime, values, label=f"Coord: {coord}", alpha=0.5, linewidth=0.8)
    
#     plt.title(f"Spatial Variation of LAI for {ilabel}")
#     plt.xlabel("Time")
#     plt.ylabel("LAI [-]")
#     plt.ylim(-0.5, 5)
#     #plt.legend(fontsize=8, loc="upper right", framealpha=0.7, ncol=2)
#     plt.tight_layout()
#     plt.show()

import seaborn as sns
color_palette = sns.color_palette("tab10", len(watershed_labels))
landcover_colors = {label: color for label, color in zip(watershed_labels, color_palette)}

for ilabel, coord_data in spatial_variation.items():
    plt.figure(figsize=(12, 6))

    # Convert the time series data into a DataFrame for easier calculation of stats
    coord_df = pd.DataFrame.from_dict(coord_data, orient='columns')  # Columns are coordinates
    coord_df.index = times_datetime

    # Calculate metrics
    mean_series = coord_df.mean(axis=1)  # Mean across coordinates
    p10_series = coord_df.quantile(0.1, axis=1)  # 10th percentile
    p90_series = coord_df.quantile(0.9, axis=1)  # 90th percentile

    # Plot the mean line
    plt.plot(
        times_datetime, 
        mean_series, 
        color=landcover_colors[ilabel], 
        label=f"{ilabel} Mean", 
        linewidth=2
    )

    # Plot the shaded area for the 10th to 90th percentile range
    plt.fill_between(
        times_datetime, 
        p10_series, 
        p90_series, 
        color=landcover_colors[ilabel], 
        alpha=0.3,  # Transparency for the shaded area
        label=f"{ilabel} 10th-90th Percentile"
    )

    # Add labels, legends, and formatting
    plt.title(f"LAI for {ilabel}: Mean and 10th-90th Percentile")
    plt.xlabel("Time")
    plt.ylabel("LAI [-]")
    plt.ylim(-0.5, 5)
    #plt.legend(fontsize=10)
    plt.grid(alpha=0.5)
    plt.tight_layout()
    plt.show()

## Process LAI_df further; crosswalk with NLCD

### Map MODIS landcover to NLCD landcover
We use NLCD for its high resolution, but need LAI from MODIS. Therefore we will get NLCD on the domain, and form the "crosswalk" between NLCD and MODIS.

[note]: 
- code largely borrowed from get_MODIS_LAI.ipynb in watershed-workflow
- to-do 1, Zhi did threshold cutting for LULC types with < 5% of the pixel coverage
- to-do 2, read LAI and LULC to standard FileManager instance in above part

In [ ]:
# get the NLCD data on the polygon
sources = watershed_workflow.source_list.get_default_sources() # only use the sources['land cover']

from shapely.ops import unary_union
watershed_polygon = unary_union(watershed_shape.geometry)  # Combine possible multi polygons into one; equavalent to watershed.exterior()

nlcd_profile, nlcd_raster = watershed_workflow.get_raster_on_shape(sources['land cover'], watershed_polygon, crs_daymet)
# nlcd = watershed_workflow.values_from_raster(surface_centroid, crs, lc_raster, lc_profile)

In [ ]:
# plot NLCD and MODIS land use
fig_nlcd, ax_nlcd = watershed_workflow.plot.get_ax(nlcd_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))

# plot the NLCD image
# -- get the NLCD colormap which uses official NLCD colors and labels
nlcd_indices, nlcd_cmap, nlcd_norm, nlcd_ticks, nlcd_labels = \
            watershed_workflow.colors.generate_nlcd_colormap(np.unique(nlcd_raster))

watershed_workflow.plot.raster(nlcd_profile, nlcd_raster, ax=ax_nlcd, cmap=nlcd_cmap, norm=nlcd_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'k', ax_nlcd)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(nlcd_raster)), cmap=nlcd_cmap, labels=nlcd_labels, ax=ax_nlcd) 
ax_nlcd.set_title("NLCD Index")

# plot the MODIS landcover
## read the MODIS with filenames
from watershed_workflow.sources.manager_modis_appeears import FileManagerMODISAppEEARS
fmodis = FileManagerMODISAppEEARS()
modislulc_data = fmodis.get_data(filenames=[fname_lulc], variables=["LULC"])

modis_raster = modislulc_data['LULC'].data[-1]
modis_profile = modislulc_data['LULC'].profile

fig_modis, ax_modis = watershed_workflow.plot.get_ax(modis_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))

# fix nan in modis_raster and modis_profile
# cause error to generate_modis_colormap and computeCrosswalkCorrelation
modis_raster = np.where(np.isnan(modis_raster), np.int16(-1), modis_raster).astype(np.int16) 
modis_profile['nodata'] = -1  # Update the nodata value in the profile
modis_profile['dtype'] = 'int16'  # Update the dtype in the profile

modis_indices, modis_cmap, modis_norm, modis_ticks, modis_labels = \
            watershed_workflow.colors.generate_modis_colormap(np.unique(modis_raster))

print(modis_indices, modis_labels)
print(modis_cmap(8))
print(modis_cmap(4))
watershed_workflow.plot.raster(modis_profile, modis_raster, ax=ax_modis, cmap=modis_cmap, norm=modis_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'k', ax_modis)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(modis_raster)), cmap=modis_cmap, labels=modis_labels, ax=ax_modis) 
ax_modis.set_title("MODIS LULC Index")

In [ ]:
dict(zip(nlcd_indices, nlcd_labels))

In [ ]:
dict(zip(modis_indices, modis_labels))

In [ ]:
# form the crosswalk and plot a correlation matrix
# crosswalk = watershed_workflow.land_cover_properties.computeCrosswalkCorrelation(modislulc_data['LULC'].profile, 
#                                                                                  modislulc_data['LULC'].data[-1],
#                                                                                  nlcd_profile, nlcd_raster)
crosswalk = watershed_workflow.land_cover_properties.computeCrosswalkCorrelation(modislulc_data['LULC'].profile, 
                                                                                 modis_raster,
                                                                                 nlcd_profile, nlcd_raster)

for key, value in crosswalk.items():
    print(f"{key}: {value}")

### Select dominant LULC types and use their LAI values for ATS run

In [ ]:
cut_threshold = 0.01

In [ ]:
lulc_ids, lulc_counts = np.unique(landcover_data_modis_masked.values[~np.isnan(landcover_data_modis_masked.values)], return_counts=True)

sum1 = sum(counts)
cutoff = sum1*cut_threshold #screen LULC types with 5% of the pixel coverage cutoff

filtered_lulc_ids = np.delete(lulc_ids, np.argwhere(lulc_counts < int(cutoff)))
print("LULC ids pass cutoff: " + str(filtered_lulc_ids))

filtered_lulc_colors = [lc_colors[i][1] for i in filtered_lulc_ids]
filtered_lulc_labels = [lc_colors[i][0] for i in filtered_lulc_ids]

print("their labels1: " + str(filtered_lulc_labels))

In [ ]:
# dominant MODIS LULC
countLULCclass = len(filtered_lulc_ids)

if(countLULCclass >= 5):
    dom = np.argpartition(-lulc_counts, range(5))[:5]
    print(lulc_ids[dom])  # prints the 5 most frequent LULC IDs
else:
    dom = np.argpartition(-lulc_counts, range(countLULCclass))[:countLULCclass]
    print(lulc_ids[dom])  # prints the most frequent LULC ID

In [ ]:
# dominant LULC and label
if(countLULCclass >= 5):
    a = lulc_ids[dom]
    LULC1 = a[0]
    LULC2 = a[1]
    LULC3 = a[2]
    LULC4 = a[3]
    LULC5 = a[4]
    LULC1label = lc_colors[LULC1][0]
    LULC2label = lc_colors[LULC2][0]
    LULC3label = lc_colors[LULC3][0]
    LULC4label = lc_colors[LULC4][0]
    LULC5label = lc_colors[LULC5][0]
elif(countLULCclass == 4):
    a = lulc_ids[dom]
    LULC1 = a[0]
    LULC2 = a[1]
    LULC3 = a[2]
    LULC4 = a[3]
    LULC1label = lc_colors[LULC1][0]
    LULC2label = lc_colors[LULC2][0]
    LULC3label = lc_colors[LULC3][0]
    LULC4label = lc_colors[LULC4][0]
elif(countLULCclass == 3):
    a = lulc_ids[dom]
    LULC1 = a[0]
    LULC2 = a[1]
    LULC3 = a[2]
    LULC1label = lc_colors[LULC1][0]
    LULC2label = lc_colors[LULC2][0]
    LULC3label = lc_colors[LULC3][0]
elif(countLULCclass == 2):
    a = lulc_ids[dom]
    LULC1 = a[0]
    LULC2 = a[1]
    LULC1label = lc_colors[LULC1][0]
    LULC2label = lc_colors[LULC2][0]
else:
    a = lulc_ids[dom]
    LULC1 = a[0]
    LULC1label = lc_colors[LULC1][0]

### MODIS and NLCD crosswalk

<font color='green'> Users may change the crosswalk between MODIS and NLCD labels based on their study area characterisctics

In [ ]:
## In general, this is the mapping to consider between NLCD LULC with MODIS LULC

In [ ]:
#Colors are based on NLCD LULC colors
MODIS_labels = ['Unclassified', 
                'Open Water', 
                'Evergreen Needleleaf Forests', 
                'Evergreen Broadleaf Forests',
                'Deciduous Needleleaf Forests', 
                'Deciduous Broadleaf Forests', 
                'Mixed Forests', 
                'Closed Shrublands', 
                'Open Shrublands', 
                'Woody Savannas', 
                'Savannas', 
                'Grasslands', 
                'Permanent Wetlands', 
                'Croplands', 
                'Urban and Built up lands', 
                'Cropland Natural Vegetation Mosaics', 
                'Permanent Snow and Ice', 
                'Barren Land', 
                'Water Bodies']

In [ ]:
NLCD_labels = ['None',
               'Open Water',
               'Evergreen Forest',
               'Evergreen Forest',
               'Deciduous Forest',
               'Deciduous Forest',
               'Mixed Forest',
               'Shrub/Scrub',
               'Shrub/Scrub',
               'Woody Wetlands',
               'Pasture/Hay',
               'Grassland/Herbaceous',
               'Emergent Herbaceous Wetlands',
               'Cultivated Crops',
               'Developed, Medium Intensity',
               'Cultivated Crops',
               'Perrenial Ice/Snow',
               'Barren Land',
               'Open Water']

NLCD_labels = [_lb.replace('/',' ') for _lb in NLCD_labels]
print(NLCD_labels) 

In [ ]:
compare_labels = pd.DataFrame({'MODIS_labels': MODIS_labels, 'NLCD_labels': NLCD_labels})
compare_labels

In [ ]:
# 1st step: change MODIS labels to NLCD labels directly

nlcd_LAI = LAI_df.copy()

if(countLULCclass >= 5):
    NLCDLULC1label=compare_labels[compare_labels['MODIS_labels']==LULC1label]
    NLCDLULC1label1 = NLCDLULC1label['NLCD_labels'].values[0]
    NLCDLULC2label=compare_labels[compare_labels['MODIS_labels']==LULC2label]
    NLCDLULC2label1 = NLCDLULC2label['NLCD_labels'].values[0]
    NLCDLULC3label=compare_labels[compare_labels['MODIS_labels']==LULC3label]
    NLCDLULC3label1 = NLCDLULC3label['NLCD_labels'].values[0]
    NLCDLULC4label=compare_labels[compare_labels['MODIS_labels']==LULC4label]
    NLCDLULC4label1 = NLCDLULC4label['NLCD_labels'].values[0]
    NLCDLULC5label=compare_labels[compare_labels['MODIS_labels']==LULC5label]
    NLCDLULC5label1 = NLCDLULC5label['NLCD_labels'].values[0]
    
    nlcd_LAI[f'NLCD {NLCDLULC1label1} LAI [-]'] = LAI_df[LULC1label]
    nlcd_LAI[f'NLCD {NLCDLULC2label1} LAI [-]'] = LAI_df[LULC2label]
    nlcd_LAI[f'NLCD {NLCDLULC3label1} LAI [-]'] = LAI_df[LULC3label]
    nlcd_LAI[f'NLCD {NLCDLULC4label1} LAI [-]'] = LAI_df[LULC4label]
    nlcd_LAI[f'NLCD {NLCDLULC5label1} LAI [-]'] = LAI_df[LULC5label]
    
    nlcd_LAI.plot(y= [f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]', f'NLCD {NLCDLULC3label1} LAI [-]', f'NLCD {NLCDLULC4label1} LAI [-]', f'NLCD {NLCDLULC5label1} LAI [-]'], lw = 1)
    plt.ylabel("LAI [-]")
    plt.xlim(datetime(2002,10,1), datetime(2025,1,25))

elif(countLULCclass == 4):
    NLCDLULC1label=compare_labels[compare_labels['MODIS_labels']==LULC1label]
    NLCDLULC1label1 = NLCDLULC1label['NLCD_labels'].values[0]
    NLCDLULC2label=compare_labels[compare_labels['MODIS_labels']==LULC2label]
    NLCDLULC2label1 = NLCDLULC2label['NLCD_labels'].values[0]
    NLCDLULC3label=compare_labels[compare_labels['MODIS_labels']==LULC3label]
    NLCDLULC3label1 = NLCDLULC3label['NLCD_labels'].values[0]
    NLCDLULC4label=compare_labels[compare_labels['MODIS_labels']==LULC4label]
    NLCDLULC4label1 = NLCDLULC4label['NLCD_labels'].values[0]
    
    nlcd_LAI[f'NLCD {NLCDLULC1label1} LAI [-]'] = LAI_df[LULC1label]
    nlcd_LAI[f'NLCD {NLCDLULC2label1} LAI [-]'] = LAI_df[LULC2label]
    nlcd_LAI[f'NLCD {NLCDLULC3label1} LAI [-]'] = LAI_df[LULC3label]
    nlcd_LAI[f'NLCD {NLCDLULC4label1} LAI [-]'] = LAI_df[LULC4label]
    
    nlcd_LAI.plot(y= [f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]', f'NLCD {NLCDLULC3label1} LAI [-]', f'NLCD {NLCDLULC4label1} LAI [-]'], lw = 1)
    plt.ylabel("LAI [-]")
    plt.xlim(datetime(2002,10,1), datetime(2025,1,25))

elif(countLULCclass == 3):
    NLCDLULC1label=compare_labels[compare_labels['MODIS_labels']==LULC1label]
    NLCDLULC1label1 = NLCDLULC1label['NLCD_labels'].values[0]
    NLCDLULC2label=compare_labels[compare_labels['MODIS_labels']==LULC2label]
    NLCDLULC2label1 = NLCDLULC2label['NLCD_labels'].values[0]
    NLCDLULC3label=compare_labels[compare_labels['MODIS_labels']==LULC3label]
    NLCDLULC3label1 = NLCDLULC3label['NLCD_labels'].values[0]
    
    nlcd_LAI[f'NLCD {NLCDLULC1label1} LAI [-]'] = LAI_df[LULC1label]
    nlcd_LAI[f'NLCD {NLCDLULC2label1} LAI [-]'] = LAI_df[LULC2label]
    nlcd_LAI[f'NLCD {NLCDLULC3label1} LAI [-]'] = LAI_df[LULC3label]
    
    nlcd_LAI.plot(y= [f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]', f'NLCD {NLCDLULC3label1} LAI [-]'], lw = 1)
    plt.ylabel("LAI [-]")
    plt.xlim(datetime(2002,10,1), datetime(2025,1,25))
    
elif(countLULCclass == 2):
    NLCDLULC1label=compare_labels[compare_labels['MODIS_labels']==LULC1label]
    NLCDLULC1label1 = NLCDLULC1label['NLCD_labels'].values[0]
    NLCDLULC2label=compare_labels[compare_labels['MODIS_labels']==LULC2label]
    NLCDLULC2label1 = NLCDLULC2label['NLCD_labels'].values[0]
    
    nlcd_LAI[f'NLCD {NLCDLULC1label1} LAI [-]'] = LAI_df[LULC1label]
    nlcd_LAI[f'NLCD {NLCDLULC2label1} LAI [-]'] = LAI_df[LULC2label]
    
    nlcd_LAI.plot(y= [f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]'], lw = 1)
    plt.ylabel("LAI [-]")
    plt.xlim(datetime(2002,10,1), datetime(2025,1,25))
    
else:
    NLCDLULC1label=compare_labels[compare_labels['MODIS_labels']==LULC1label]
    NLCDLULC1label1 = NLCDLULC1label['NLCD_labels'].values[0]
    
    nlcd_LAI[f'NLCD {NLCDLULC1label1} LAI [-]'] = LAI_df[LULC1label]
    
    nlcd_LAI.plot(y= [f'NLCD {NLCDLULC1label1} LAI [-]'], lw = 1)
    plt.ylabel("LAI [-]")
    plt.xlim(datetime(2002,10,1), datetime(2025,1,25))

In [ ]:
LULC1label, LULC2label, LULC3label, LULC4label, LULC5label

In [ ]:
NLCDLULC1label1, NLCDLULC2label1, NLCDLULC3label1, NLCDLULC4label1, NLCDLULC5label1

In [ ]:
# 2nd step: adjust NLCD labels above based on NLCD map
# for Naches here,
# - keep Evergreen Forest, Grassland Herbaceous, and Woody Wetlands
# - drop Pasture Hay and Cultivated Crops

# for i in [f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]', f'NLCD {NLCDLULC3label1} LAI [-]', f'NLCD {NLCDLULC4label1} LAI [-]']:
#     if i == f'NLCD {NLCDLULC2label1} LAI [-]':
#         a = nlcd_LAI[i].values
#         print(a)
#     if i == f'NLCD {NLCDLULC3label1} LAI [-]':
#         b = nlcd_LAI[i].values
#         print(b)
# c = (a+b)/2
# print(c)

### ~~Save processed LAI with NLCD labels to hdf5~~

In [ ]:
# outputs['modis_filename_watershed_raw'] = f'../data-processed/{watershed_name}/{watershed_name}_MODIS_LAI_20020704_20250124.h5'

# with h5.File(outputs['modis_filename_watershed_raw'], 'w') as fout:
#     for i in ['time [s]', f'NLCD {NLCDLULC1label1} LAI [-]', f'NLCD {NLCDLULC2label1} LAI [-]', f'NLCD {NLCDLULC3label1} LAI [-]', f'NLCD {NLCDLULC4label1} LAI [-]']:
#         if i == f'NLCD {NLCDLULC2label1} LAI [-]':
#             fout.create_dataset('NLCD Shrub Scrub LAI [-]', data=c)
#         elif i == f'NLCD {NLCDLULC3label1} LAI [-]':
#             continue
#         else:
#             fout.create_dataset(i, data=nlcd_LAI[i].values)

# Get MODIS-LAI for the hillslope site

## Plot NLCD and MODIS-LULC with 2D transect

In [ ]:
m2_mat_filename =  f'../data-processed/{site_name}/startendcoords_{site_name}.mat'
loaded_data  = loadmat(m2_mat_filename)
start_coords = loaded_data['start_coords'].flatten()
end_coords   = loaded_data['end_coords'].flatten()

print(start_coords)
print(end_coords)

In [ ]:
# Bounding box in crs_daymet
dx = 5000/4
dy = 4000/4
xmin = (start_coords[0]+end_coords[0])/2 - dx/2
xmax = (start_coords[0]+end_coords[0])/2 + dx/2
ymin = (start_coords[1]+end_coords[1])/2 - dy/2
ymax = (start_coords[1]+end_coords[1])/2 + dy/2
print("Bounding box in crs_daymet: "+ str([xmin, xmax, ymin, ymax]))

# Determine Bounding box in nlcd_profile['crs']
xmin_nlcd, ymin_nlcd = pyproj.transform(crs_daymet, nlcd_profile['crs'], xmin, ymin)
xmax_nlcd, ymax_nlcd = pyproj.transform(crs_daymet, nlcd_profile['crs'], xmax, ymax)
print("Bounding box in nlcd_profile['crs']: "+ str([xmin_nlcd, xmax_nlcd, ymin_nlcd, ymax_nlcd]))
# Determine start_coods and end_coords in nlcd_profile['crs']
start_coords_nlcd, end_coords_nlcd = np.zeros(2), np.zeros(2)
start_coords_nlcd[0], start_coords_nlcd[1] = pyproj.transform(crs_daymet, nlcd_profile['crs'], start_coords[0], start_coords[1])
end_coords_nlcd[0],   end_coords_nlcd[1]   = pyproj.transform(crs_daymet, nlcd_profile['crs'], end_coords[0], end_coords[1])

# Determine Bounding box in modis_profile['crs']
xmin_modis, ymin_modis = pyproj.transform(crs_daymet, modis_profile['crs'], xmin, ymin)
xmax_modis, ymax_modis = pyproj.transform(crs_daymet, modis_profile['crs'], xmax, ymax)
print("Bounding box in modis_profile['crs']: "+ str([xmin_modis, xmax_modis, ymin_modis, ymax_modis]))

In [ ]:
# plot NLCD
fig_nlcd, ax_nlcd = watershed_workflow.plot.get_ax(nlcd_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
watershed_workflow.plot.raster(nlcd_profile, nlcd_raster, ax=ax_nlcd, cmap=nlcd_cmap, norm=nlcd_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'k', ax_nlcd)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(nlcd_raster)), cmap=nlcd_cmap, labels=nlcd_labels, ax=ax_nlcd) 
watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'r', ax_nlcd)
ax_nlcd.set_title("NLCD Index")

In [ ]:
# plot NLCD - zoom in
fig_nlcd, ax_nlcd = watershed_workflow.plot.get_ax(nlcd_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
watershed_workflow.plot.raster(nlcd_profile, nlcd_raster, ax=ax_nlcd, cmap=nlcd_cmap, norm=nlcd_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'k', ax_nlcd)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(nlcd_raster)), cmap=nlcd_cmap, labels=nlcd_labels, ax=ax_nlcd) 
watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'r', ax_nlcd)
ax_nlcd.set_title("NLCD Index - zoom in")
ax_nlcd.set_xlim(xmin_nlcd, xmax_nlcd)
ax_nlcd.set_ylim(ymin_nlcd, ymax_nlcd)

In [ ]:
# plot MODIS landcover
fig_modis, ax_modis = watershed_workflow.plot.get_ax(modis_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
watershed_workflow.plot.raster(modis_profile, modis_raster, ax=ax_modis, cmap=modis_cmap, norm=modis_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'k', ax_modis)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(modis_raster)), cmap=modis_cmap, labels=modis_labels, ax=ax_modis) 
watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'r', ax_modis)
ax_modis.set_title("MODIS LULC Index")

In [ ]:
# plot MODIS landcover - zoom in
fig_modis, ax_modis = watershed_workflow.plot.get_ax(modis_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
watershed_workflow.plot.raster(modis_profile, modis_raster, ax=ax_modis, cmap=modis_cmap, norm=modis_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'k', ax_modis)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(modis_raster)), cmap=modis_cmap, labels=modis_labels, ax=ax_modis) 
watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'r', ax_modis)
ax_modis.set_title("MODIS LULC Index - zoom in")
ax_modis.set_xlim(ymin_modis, ymax_modis)
ax_modis.set_ylim(xmin_modis, xmax_modis)

## Config the mapping between NLCD LULC and MODIS LULC

In [ ]:
# existing NLCD LULC in current watershed
dict(zip(nlcd_indices, nlcd_labels))

In [ ]:
# existing MODIS LULC in current watershed
dict(zip(modis_indices, modis_labels))

In [ ]:
# SELECTED dominant MODIS LULC, based on lulc_ids[dom]
# for Naches, select 1_Evergreen Needleleaf Forests, 8_Woody Savannas, and 10_Grassland
modis_lulc_ids_selected = lulc_ids[dom][[0,1,2]]

modis_dict = dict(zip(modis_indices, modis_labels))
modis_dict_subset = {idx: modis_dict[idx] for idx in modis_lulc_ids_selected if idx in modis_dict}
modis_dict_subset

## Extract NLCD labels for each hillslope cells

In [ ]:
from shapely.geometry import LineString, box
import rasterio.transform

# Create the line geometry
line_geom = LineString([start_coords_nlcd, end_coords_nlcd])

# Get transform and pixel size
transform = nlcd_profile['transform']
pixel_width = transform[0]
pixel_height = -transform[4]

# Get the bounding box of the line
line_bounds = line_geom.bounds  # (minx, miny, maxx, maxy)

# Convert bounds to pixel coordinates
row_min, col_min = rasterio.transform.rowcol(transform, line_bounds[0], line_bounds[3])
row_max, col_max = rasterio.transform.rowcol(transform, line_bounds[2], line_bounds[1])

# Ensure we're within raster bounds
row_min = max(0, row_min - 1)
row_max = min(nlcd_raster.shape[0], row_max + 2)
col_min = max(0, col_min - 1)
col_max = min(nlcd_raster.shape[1], col_max + 2)

# Find all pixels that intersect with the line
rr = []
cc = []
nlcd_values_sampled = []

for r in range(row_min, row_max):
    for c in range(col_min, col_max):
        # Get pixel bounds in geographic coordinates
        x_left, y_top = transform * (c, r)
        x_right = x_left + pixel_width
        y_bottom = y_top - pixel_height
        
        # Create pixel polygon
        pixel_box = box(x_left, y_bottom, x_right, y_top)
        
        # Check if line intersects this pixel
        if line_geom.intersects(pixel_box):
            rr.append(r)
            cc.append(c)
            nlcd_values_sampled.append(nlcd_raster[r, c])

# Convert to numpy arrays
rr = np.array(rr)
cc = np.array(cc)
nlcd_values_sampled = np.array(nlcd_values_sampled)

print(f"Number of pixels crossed: {len(nlcd_values_sampled)}")
print(f"NLCD values along line: {nlcd_values_sampled}")
print(f"Unique NLCD values: {np.unique(nlcd_values_sampled)}")

In [ ]:
nlcd_dict = dict(zip(nlcd_indices, nlcd_labels))
nlcd_dict_subset = {idx: nlcd_dict[idx] for idx in np.unique(nlcd_values_sampled) if idx in nlcd_dict}
nlcd_dict_subset

In [ ]:
# NLCD to MODIS mapping, this works for the whole Naches
nlcd_to_modis_map = {
    42: 1,   # Evergreen Forest -> Evergreen Needleleaf Forests
    41: 8,   # Deciduous Forest -> Woody Savannas
    43: 8,   # Mixed Forest -> Woody Savannas
    90: 8,   # Woody Wetlands -> Woody Savannas
    52: 10,  # Shrub/Scrub -> Grasslands
    71: 10,  # Grassland/Herbaceous -> Grasslands
    81: 10,  # Pasture/Hay -> Grasslands
    82: 10,  # Cultivated Crops -> Grasslands
    95: 10   # Emergent Herbaceous Wetlands -> Grasslands
}
## although for this specific hillslope, we only need the map below.
# nlcd_to_modis_map = {
#     42: 1,   # Evergreen Forest -> Evergreen Needleleaf Forests
#     52: 10,  # Shrub/Scrub -> Grasslands
# }

In [ ]:
# plot NLCD - zoom in with crossed pixels highlighted
from matplotlib.patches import Rectangle

fig_nlcd, ax_nlcd = watershed_workflow.plot.get_ax(nlcd_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
watershed_workflow.plot.raster(nlcd_profile, nlcd_raster, ax=ax_nlcd, cmap=nlcd_cmap, norm=nlcd_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'k', ax_nlcd)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(nlcd_raster)), cmap=nlcd_cmap, labels=nlcd_labels, ax=ax_nlcd) 
watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, nlcd_profile['crs']), nlcd_profile['crs'], 'r', ax_nlcd)

# Add white edges for crossed pixels
transform = nlcd_profile['transform']
pixel_width = transform[0]
pixel_height = -transform[4]  # negative because y goes down

for r, c in zip(rr, cc):
    # Get the geographic coordinates of the pixel's upper-left corner
    x, y = transform * (c, r)
    
    # Add a rectangle with white edge and no fill
    rect = Rectangle((x, y - pixel_height), pixel_width, pixel_height,
                     linewidth=1.5, edgecolor='white', facecolor='none')
    ax_nlcd.add_patch(rect)

ax_nlcd.set_title("NLCD Index - zoom in with crossed pixels")
ax_nlcd.set_xlim(xmin_nlcd, xmax_nlcd)
ax_nlcd.set_ylim(ymin_nlcd, ymax_nlcd)

In [ ]:
# Step 1: Get coordinates of the crossed NLCD pixels
nlcd_transform = nlcd_profile['transform']
nlcd_pixel_coords = []

for r, c in zip(rr, cc):
    # Get pixel center in NLCD coordinate system
    x_nlcd, y_nlcd = nlcd_transform * (c + 0.5, r + 0.5)
    nlcd_pixel_coords.append((x_nlcd, y_nlcd))

print(f"Number of NLCD pixels: {len(nlcd_pixel_coords)}")
print("First few NLCD pixel coordinates:")
for i in range(min(5, len(nlcd_pixel_coords))):
    print(f"  Pixel {i}: ({nlcd_pixel_coords[i][0]:.2f}, {nlcd_pixel_coords[i][1]:.2f})")

# Step 2: Transform NLCD pixel coordinates to MODIS coordinate system
nlcd_to_modis_coords = []

for x_nlcd, y_nlcd in nlcd_pixel_coords:
    # Transform from NLCD CRS to Daymet CRS (intermediate step)
    x_daymet, y_daymet = pyproj.transform(nlcd_profile['crs'], crs_daymet, x_nlcd, y_nlcd)
    
    # Transform from Daymet CRS to MODIS CRS
    x_modis, y_modis = pyproj.transform(crs_daymet, modis_profile['crs'], x_daymet, y_daymet)
    
    nlcd_to_modis_coords.append((x_modis, y_modis))

print(f"\nNLCD pixel coordinates transformed to MODIS CRS:")
for i in range(min(5, len(nlcd_to_modis_coords))):
    print(f"  Pixel {i}: ({nlcd_to_modis_coords[i][0]:.2f}, {nlcd_to_modis_coords[i][1]:.2f})")

# Optional: Convert to numpy array for easier manipulation
nlcd_pixel_coords = np.array(nlcd_pixel_coords)
nlcd_to_modis_coords = np.array(nlcd_to_modis_coords)

print(f"\nShape of coordinate arrays: {nlcd_to_modis_coords.shape}")

## Find closest MODIS grid with corresponding MODIS LULC labels

- the LAI NaN mask is not completed.

In [ ]:
# plot MODIS landcover - zoom in with NLCD pixel centers
fig_modis, ax_modis = watershed_workflow.plot.get_ax(modis_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
watershed_workflow.plot.raster(modis_profile, modis_raster, ax=ax_modis, cmap=modis_cmap, norm=modis_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'k', ax_modis)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(modis_raster)), cmap=modis_cmap, labels=modis_labels, ax=ax_modis) 
watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'r', ax_modis)

# Plot the 29 NLCD pixel centers transformed to MODIS coordinates
ax_modis.scatter(nlcd_to_modis_coords[:, 1], nlcd_to_modis_coords[:, 0], 
                 facecolors='none', s=50, edgecolors='gray', linewidths=1.5, 
                 marker='o', zorder=10, label='NLCD pixel centers')

ax_modis.set_title("MODIS LULC Index - zoom in with NLCD pixels")
ax_modis.set_xlim(ymin_modis, ymax_modis)
ax_modis.set_ylim(xmin_modis, xmax_modis)
ax_modis.legend(loc='upper right')

In [ ]:
## The LAI NaN mask is not completed, and perhaps is not necessary

# Load unclipped LAI data to match modis_raster spatial grid
dset_lai = xr.open_dataset(fname_lai)
LAI_data_unclipped = dset_lai.Lai_500m

# # Create valid LAI mask from unclipped data
# # This will now have the same shape as modis_raster
# valid_lai_mask = ~np.all(np.isnan(LAI_data_unclipped.values), axis=0)
# print(f"Total MODIS pixels: {valid_lai_mask.size}")
# print(f"Pixels with valid LAI: {np.sum(valid_lai_mask)}")
# print(f"Pixels with all NaN LAI: {np.sum(~valid_lai_mask)}")

# # Verify shapes match
# print(f"modis_raster shape: {modis_raster.shape}")
# print(f"valid_lai_mask shape: {valid_lai_mask.shape}")
# print(f"LAI_data_unclipped shape: {LAI_data_unclipped.shape}")

In [ ]:
# from matplotlib.patches import Rectangle

# # plot MODIS landcover - zoom in with NLCD pixel centers
# fig_modis, ax_modis = watershed_workflow.plot.get_ax(modis_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
# watershed_workflow.plot.raster(modis_profile, modis_raster, ax=ax_modis, cmap=modis_cmap, norm=modis_norm)
# watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'k', ax_modis)
# watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(modis_raster)), cmap=modis_cmap, labels=modis_labels, ax=ax_modis) 
# watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'r', ax_modis)

# # Add red borders to all pixels with NaN LAI
# modis_transform = modis_profile['transform']
# pixel_width = modis_transform[0]
# pixel_height = -modis_transform[4]

# rows, cols = np.where(~valid_lai_mask)
# for r, c in zip(rows, cols):
#     x, y = modis_transform * (c, r)
#     rect = Rectangle((x, y), pixel_height, pixel_width,
#                    linewidth=1.5, edgecolor='red', 
#                    facecolor='none', zorder=5)
#     ax_modis.add_patch(rect)

# # Plot the 29 NLCD pixel centers transformed to MODIS coordinates
# ax_modis.scatter(nlcd_to_modis_coords[:, 1], nlcd_to_modis_coords[:, 0], 
#                  facecolors='none', s=50, edgecolors='gray', linewidths=1.5, 
#                  marker='o', zorder=10, label='NLCD pixel centers')

# ax_modis.set_title("MODIS LULC Index - zoom in (red border = NaN LAI)")
# ax_modis.set_xlim(ymin_modis, ymax_modis)
# ax_modis.set_ylim(xmin_modis, xmax_modis)
# ax_modis.legend(loc='upper right')

In [ ]:
print(modis_raster.shape)
print(LAI_data.data.shape)
print(LAI_data_unclipped.data.shape)

In [ ]:
from scipy.spatial import cKDTree

# Prepare MODIS grid information
modis_transform = modis_profile['transform']
modis_pixel_width = modis_transform[0]
modis_pixel_height = -modis_transform[4]

# Build a spatial index of MODIS pixel centers for each MODIS LULC type
modis_pixel_centers = {}  # key: modis_lulc_idx, value: dict with 'coords' and 'indices'

for modis_idx in np.unique(modis_raster):
    if modis_idx == -1:  # skip nodata
        continue
    
    # Find all pixels with this MODIS LULC type
    rows, cols = np.where(modis_raster == modis_idx)
    
    if len(rows) > 0:
        # Get center coordinates for each pixel in MODIS coordinate system
        centers = []
        for r, c in zip(rows, cols):
            x, y = modis_transform * (c + 0.5, r + 0.5)  # pixel center
            centers.append([x, y])  # store as [x, y]
        
        # Store coordinates and build KDTree for fast nearest neighbor search
        modis_pixel_centers[modis_idx] = {
            'coords': np.array(centers),
            'row_col': list(zip(rows, cols)),
            'tree': cKDTree(centers)
        }

# Now loop through crossed NLCD pixels using the transformed coordinates
nlcd_to_modis_results = []

for i, (r, c) in enumerate(zip(rr, cc)):
    nlcd_idx = nlcd_values_sampled[i]
    
    # Get NLCD pixel center in MODIS coordinate system (already transformed)
    # nlcd_to_modis_coords is stored as [x, y] where index 0=x, index 1=y
    x_modis, y_modis = nlcd_to_modis_coords[i, 0], nlcd_to_modis_coords[i, 1]
    
    # Check if this NLCD index has a mapping to MODIS
    if nlcd_idx in nlcd_to_modis_map:
        modis_idx = nlcd_to_modis_map[nlcd_idx]
        
        # Check if we have MODIS pixels with this LULC type
        if modis_idx in modis_pixel_centers:
            # Find closest MODIS pixel with this LULC type
            tree = modis_pixel_centers[modis_idx]['tree']
            distance, nearest_idx = tree.query([y_modis, x_modis])  # query with [x, y]
            
            # Get the row, col of the nearest MODIS pixel
            modis_r, modis_c = modis_pixel_centers[modis_idx]['row_col'][nearest_idx]
            
            nlcd_to_modis_results.append({
                'nlcd_pixel': (r, c),
                'nlcd_idx': nlcd_idx,
                'nlcd_coords_modis': (x_modis, y_modis),  # store as (x, y)
                'modis_idx': modis_idx,
                'modis_pixel': (modis_r, modis_c),
                'distance': distance
            })
            
            print(f"NLCD pixel ({r},{c}) [idx={nlcd_idx}] at ({x_modis:.2f}, {y_modis:.2f}) "
                  f"-> MODIS idx={modis_idx}, pixel ({modis_r},{modis_c}), distance={distance:.2f}")
        else:
            print(f"NLCD pixel ({r},{c}) [idx={nlcd_idx}] -> MODIS idx={modis_idx} NOT FOUND in raster")
    else:
        print(f"NLCD pixel ({r},{c}) [idx={nlcd_idx}] -> NO MAPPING to MODIS")

# Convert to DataFrame for easier analysis
import pandas as pd
results_df = pd.DataFrame(nlcd_to_modis_results)
print(f"\nTotal mapped pixels: {len(nlcd_to_modis_results)}")
print(results_df)

In [ ]:
# plot MODIS landcover - zoom in with NLCD pixel centers
fig_modis, ax_modis = watershed_workflow.plot.get_ax(modis_profile['crs'], nrow=1, ncol=1, index=1, figsize=(15,5))
watershed_workflow.plot.raster(modis_profile, modis_raster, ax=ax_modis, cmap=modis_cmap, norm=modis_norm)
watershed_workflow.plot.shply(watershed_workflow.warp.shply(watershed_polygon, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'k', ax_modis)
watershed_workflow.colors.colorbar_index(ncolors=len(np.unique(modis_raster)), cmap=modis_cmap, labels=modis_labels, ax=ax_modis) 
watershed_workflow.plot.shply(watershed_workflow.warp.shply(xsec_plg_dict_shply, crs_daymet, modis_profile['crs']), modis_profile['crs'], 'r', ax_modis)

# Plot the 29 NLCD pixel centers transformed to MODIS coordinates
ax_modis.scatter(nlcd_to_modis_coords[:, 1], nlcd_to_modis_coords[:, 0], 
                 facecolors='none', s=50, edgecolors='gray', linewidths=1.5, 
                 marker='o', zorder=10, label='NLCD pixel centers')

# Get MODIS pixel centers and draw connections
modis_transform = modis_profile['transform']
for result in nlcd_to_modis_results:
    # NLCD coords in MODIS CRS
    x_nlcd, y_nlcd = result['nlcd_coords_modis']
    
    # MODIS pixel center coords
    modis_r, modis_c = result['modis_pixel']
    x_modis, y_modis = modis_transform * (modis_c + 0.5, modis_r + 0.5)
    
    # Draw dashed line connecting them
    ax_modis.plot([y_nlcd, x_modis], [x_nlcd, y_modis],
                  '--', linewidth=1, color='gray', alpha=0.7, zorder=9)

# Mark MODIS pixel centers with crosses
modis_centers_x = []
modis_centers_y = []
for result in nlcd_to_modis_results:
    modis_r, modis_c = result['modis_pixel']
    x_modis, y_modis = modis_transform * (modis_c + 0.5, modis_r + 0.5)
    modis_centers_x.append(x_modis)
    modis_centers_y.append(y_modis)

ax_modis.scatter(modis_centers_x, modis_centers_y, 
                 c='gray', s=80, marker='x', linewidths=2, 
                 zorder=11, label='MODIS matched pixels')

ax_modis.set_title("MODIS LULC Index - zoom in with NLCD-MODIS mapping")
ax_modis.set_xlim(ymin_modis-(ymax_modis-ymin_modis)*0.1, ymax_modis+(ymax_modis-ymin_modis)*0.1)
ax_modis.set_ylim(xmin_modis-(xmax_modis-xmin_modis)*0.1, xmax_modis+(xmax_modis-xmin_modis)*0.1)
ax_modis.legend(loc='upper right')

## Calculate the averaged MODIS-LAI

- sometimes, multiple MODIS grids with the same MODIS LULC will be identified for a hillslope

In [ ]:
# Group MODIS pixels by MODIS LULC index
from collections import defaultdict

modis_pixels_by_lulc = defaultdict(list)
for result in nlcd_to_modis_results:
    modis_idx = result['modis_idx']
    modis_pixel = result['modis_pixel']
    modis_pixels_by_lulc[modis_idx].append(modis_pixel)

# Print summary
print("Summary of matched MODIS pixels by LULC type:")
for modis_idx, pixels in modis_pixels_by_lulc.items():
    unique_pixels = list(set(pixels))  # Remove duplicates
    modis_label = dict(zip(modis_indices, modis_labels))[modis_idx]
    print(f"  MODIS idx {modis_idx} ({modis_label}): {len(unique_pixels)} unique pixels")
    print(f"    Pixels: {unique_pixels}")

In [ ]:
# Extract LAI time series for each matched MODIS pixel and calculate average per LULC type
times = LAI_data_unclipped.time.values
LAI_hillslope = []

for itime in times:
    iCC_LAI = LAI_data_unclipped.sel(time=itime).values
    iLAI_lulc = {}
    
    for modis_idx, pixels in modis_pixels_by_lulc.items():
        # Get unique pixels (in case same pixel matched multiple NLCD pixels)
        unique_pixels = list(set(pixels))
        
        # Extract LAI values for all matched pixels
        lai_values = []
        for modis_r, modis_c in unique_pixels:
            lai_val = iCC_LAI[modis_r, modis_c]
            if not np.isnan(lai_val):
                lai_values.append(lai_val)

        # Calculate mean LAI for this MODIS LULC type
        if len(lai_values) > 0:
            iLAI_lulc[modis_idx] = np.mean(lai_values)
        else:
            iLAI_lulc[modis_idx] = np.nan
    
    LAI_hillslope.append(iLAI_lulc)

# Convert to DataFrame similar to LAI_df
modis_lulc_labels = [dict(zip(modis_indices, modis_labels))[idx] for idx in sorted(modis_pixels_by_lulc.keys())]
modis_lulc_indices = sorted(modis_pixels_by_lulc.keys())

LAI_hillslope_df = pd.DataFrame([
    [time_data[idx] for idx in modis_lulc_indices] 
    for time_data in LAI_hillslope
], columns=modis_lulc_labels)

LAI_hillslope_df['datetime'] = LAI_data_unclipped.indexes['time'].to_datetimeindex()
LAI_hillslope_df = LAI_hillslope_df.iloc[0:]
LAI_hillslope_df.set_index('datetime', inplace=True)
LAI_hillslope_df['time [s]'] = (LAI_hillslope_df.index - LAI_hillslope_df.index[0]).total_seconds()

print("\nLAI DataFrame for hillslope:")
print(LAI_hillslope_df)

# Plot
LAI_hillslope_df.drop(columns=['time [s]']).plot()
plt.ylabel('LAI [-]')
plt.title('Hillslope LAI by MODIS LULC Type')
plt.xlim(datetime(2002,10,1), datetime(2025,1,25))
plt.ylim(-0.5, 5)
plt.show()

In [ ]:
LAI_hillslope_df

In [ ]:
# Create a mapping from MODIS labels to NLCD labels using compare_labels
modis_to_nlcd_label_map = {}
for modis_label in modis_lulc_labels:
    # Find the corresponding NLCD label from compare_labels
    matching_row = compare_labels[compare_labels['MODIS_labels'] == modis_label]
    if not matching_row.empty:
        nlcd_label = matching_row['NLCD_labels'].values[0]
        modis_to_nlcd_label_map[modis_label] = nlcd_label
    else:
        # If no match found, keep the MODIS label
        modis_to_nlcd_label_map[modis_label] = modis_label

print("MODIS to NLCD label mapping:")
print(modis_to_nlcd_label_map)

# Create new column names with NLCD labels
new_column_names = {modis_label: f'NLCD {nlcd_label} LAI [-]' 
                    for modis_label, nlcd_label in modis_to_nlcd_label_map.items()}

# Don't rename 'time [s]' column
new_column_names['time [s]'] = 'time [s]'

print("\nNew column names:")
print(new_column_names)

# Rename the columns
LAI_hillslope_df_nlcd = LAI_hillslope_df.rename(columns=new_column_names)

# Display the renamed dataframe
print("\nRenamed DataFrame:")
print(LAI_hillslope_df_nlcd)

# Plot with new labels
LAI_hillslope_df_nlcd.drop(columns=['time [s]']).plot()
plt.ylabel('LAI [-]')
plt.title('Hillslope LAI by NLCD LULC Type')
plt.xlim(datetime(2002,10,1), datetime(2025,1,25))
plt.ylim(-0.5, 5)
plt.show()

In [ ]:
# 2nd step: further adjust NLCD labels above
nlcd_dict_subset
# nlcd_to_modis_map = {
#     42: 1,   # Evergreen Forest -> Evergreen Needleleaf Forests
#     52: 10,  # Shrub/Scrub -> Grassland Herbaceous
# }
# -> need to rename Grassland Herbaceous to Shrub Scrub
# double check nlcd_crosswalk_modis in 1a-main_workflow_Naches.ats1.5.ipynb
# nlcd_crosswalk_modis = {'Evergreen Forest' : 'NLCD Evergreen Forest',
#                         'Shrub/Scrub'      : 'NLCD Shrub Scrub'}

In [ ]:
LAI_hillslope_df_nlcd.rename(columns={
    'NLCD Grassland Herbaceous LAI [-]': 'NLCD Shrub Scrub LAI [-]'
}, inplace=True)

# Plot with new labels
LAI_hillslope_df_nlcd.drop(columns=['time [s]']).plot()
plt.ylabel('LAI [-]')
plt.title('Hillslope LAI by NLCD LULC Type')
plt.xlim(datetime(2002,10,1), datetime(2025,1,25))
plt.ylim(-0.5, 5)
plt.show()

### Despike native MODIS observations

Remove isolated temporal spikes and implausible one- or two-observation high-LAI runs before saving the local hillslope LAI series and interpolating it to daily forcing. Inspect the later **MODIS spinup raw** and **MODIS transient raw** plots, then adjust `despike_threshold` and `short_high_run_floor` if needed and rerun from this section. A run is replaced only when both outer neighboring observations are finite and the entire run exceeds them by more than the despike threshold.


In [ ]:
# Despike native MODIS observations before daily interpolation.
# Both thresholds are selected interactively from the later "MODIS spinup raw"
# and "MODIS transient raw" plots.  The high-run rule catches the remaining
# implausible >6 LAI one- or two-observation spikes without capping sustained LAI.
despike_threshold = 2.0  # LAI [-]; deviation above both neighboring observations
short_high_run_floor = 6.0  # LAI [-]; inspect and revise from the raw phase plots

def despike_native_modis(frame, threshold, high_run_floor):
    corrected = frame.copy()
    lai_columns = [column for column in corrected.columns if column.startswith('NLCD ') and column.endswith(' LAI [-]')]
    replacements = []
    for column in lai_columns:
        values = corrected[column].copy()
        previous = values.shift(1)
        following = values.shift(-1)
        neighbor_mean = (previous + following) / 2.0
        isolated_spike = (
            values.notna() & previous.notna() & following.notna() &
            ((previous - following).abs() <= threshold) &
            ((values - neighbor_mean).abs() > threshold)
        )
        for position in np.flatnonzero(isolated_spike.to_numpy()):
            replacements.append((values.index[position], column, values.iloc[position], neighbor_mean.iloc[position], 'isolated'))
            values.iloc[position] = neighbor_mean.iloc[position]

        position = 0
        while position < len(values):
            if not pd.notna(values.iloc[position]) or values.iloc[position] < high_run_floor:
                position += 1
                continue
            run_end = position
            while run_end + 1 < len(values) and pd.notna(values.iloc[run_end + 1]) and values.iloc[run_end + 1] >= high_run_floor:
                run_end += 1
            run_length = run_end - position + 1
            if position > 0 and run_end + 1 < len(values) and run_length <= 2:
                left_value = values.iloc[position - 1]
                right_value = values.iloc[run_end + 1]
                run_values = values.iloc[position:run_end + 1]
                if pd.notna(left_value) and pd.notna(right_value) and (run_values > max(left_value, right_value) + threshold).all():
                    for offset in range(run_length):
                        old_value = values.iloc[position + offset]
                        new_value = left_value + (right_value - left_value) * (offset + 1) / (run_length + 1)
                        replacements.append((values.index[position + offset], column, old_value, new_value, 'short high run'))
                        values.iloc[position + offset] = new_value
            position = run_end + 1
        corrected[column] = values
    return corrected, replacements

LAI_hillslope_df_nlcd, despike_replacements = despike_native_modis(
    LAI_hillslope_df_nlcd, despike_threshold, short_high_run_floor
)
print(f'Despike threshold: {despike_threshold:.2f} LAI; short high-run floor: {short_high_run_floor:.2f} LAI; replacements: {len(despike_replacements)}')
for timestamp, column, old_value, new_value, method in despike_replacements:
    print(f'  {timestamp:%Y-%m-%d} | {column} ({method}): {old_value:.3f} -> {new_value:.3f}')


## Save processed 'local' LAI with NLCD labels to hdf5

In [ ]:
outputs['modis_filename_site_raw'] = f'../data-processed/{site_name}/{site_name}_MODIS_LAI_20020704_20260113.h5'

with h5.File(outputs['modis_filename_site_raw'], 'w') as fout:
    # Save 'time [s]' column
    fout.create_dataset('time [s]', data=LAI_hillslope_df_nlcd['time [s]'].values)
    
    # Save all NLCD LAI columns
    for col in LAI_hillslope_df_nlcd.columns:
        if col != 'time [s]':  # Already saved above
            fout.create_dataset(col, data=LAI_hillslope_df_nlcd[col].values)

## Process LAI data with NLCD labels for ATS

- interpolate every 4d to daily
- remove leap year (same as DayMet, remove 12/31 if it's a leap year)
- smooth
- typical year

### interpolate, remove leap year, and smooth

In [ ]:
# use data from start_year_spinup to end_year_spinup, to generate typical year data for 1) steady-state spinup and 2) cyclic spinup. 
# 1) cyclic for nyears_steadystate_spinup years for steady-state spinup
# 2) cyclic for nyears_cyclic_spinup years for cyclic spinup
# start_year_spinup         = 2005 # in config.json
# end_year_spinup           = 2015
# nyears_steadystate_spinup = 10
# nyears_cyclic_spinup      = 10
# start_year_transient      = 2016
# end_year_transient        = 2020

In [ ]:
d = h5.File(outputs['modis_filename_site_raw'],'r')
df = pd.DataFrame()
for k in d.keys():
    df[k] = d[k][:]
df['time [d]'] = df['time [s]']/86400

df

In [ ]:
# interpolate this time series into a daily time series
# ts = np.arange(8214, 14600, 1)
ts = np.arange(df['time [d]'].values[-1]+1)
df_interp = pd.DataFrame()
df_interp['time [d]'] = ts

for k in df.keys():
    if k != 'time [s]':
        f = scipy.interpolate.interp1d(df['time [d]'][:], df[k][:])
        df_interp[k] = f(ts)

df = df_interp
#df['datetime'] = pd.to_datetime(df['time [d]'], unit='D', origin=pd.Timestamp('2002-10-01'))
origin_str = LAI_data.time[0].item().strftime('%Y-%m-%d')
df['datetime'] = pd.to_datetime(df['time [d]'], unit='D', origin=origin_str)

df

In [ ]:
def crop_phase(frame, dates, name):
    start, end = dates[0].isoformat(), dates[-1].isoformat(); out = frame.loc[(frame['datetime'] >= start) & (frame['datetime'] <= end)].reset_index(drop=True).copy(); out['time [d]'] = (out['datetime'] - pd.to_datetime(start)).dt.days; print(f'{name}: {start}..{end}, {len(out)} Gregorian days'); return out
startdate_spinup, enddate_spinup = spinup_dates[0].isoformat(), spinup_dates[-1].isoformat(); startdate_transient, enddate_transient = prefire_dates[0].isoformat(), prefire_dates[-1].isoformat()
df_cropped_spinup = crop_phase(df, spinup_dates, 'Spinup MODIS crop'); df_cropped_transient = crop_phase(df, prefire_dates, 'Prefire MODIS crop'); df_cropped_postfire = crop_phase(df, postfire_dates, 'Postfire MODIS crop') if postfire_dates else None


In [ ]:
def remove_leap_days(frame, expected, name):
    out = frame.loc[~((frame.datetime.dt.month == 2) & (frame.datetime.dt.day == 29))].reset_index(drop=True).copy()
    if len(out) != expected: raise ValueError(f'{name}: expected {expected}, found {len(out)}')
    out['time [d]'] = np.arange(len(out)); return out.drop(columns=['datetime'])
df_cropped_spinup = remove_leap_days(df_cropped_spinup, len(spinup_dates), 'spinup'); df_cropped_transient = remove_leap_days(df_cropped_transient, len(prefire_dates), 'prefire')
if postfire_dates: df_cropped_postfire = remove_leap_days(df_cropped_postfire, len(postfire_dates), 'postfire')


In [ ]:
def smooth_phase(frame):
    out = pd.DataFrame({'time [d]': frame['time [d]']})
    for key in frame:
        if key != 'time [d]': out[key] = scipy.signal.savgol_filter(frame[key], 101, 3)
    return out
df_cropped_spinup_smooth = smooth_phase(df_cropped_spinup); df_cropped_transient_smooth = smooth_phase(df_cropped_transient); df_cropped_postfire_smooth = smooth_phase(df_cropped_postfire) if postfire_dates else None


### Save MODIS - write to disk

The prefire and optional postfire plots below show the final smoothed phase-local LAI series that will be written to HDF5.

- for `df_cropped_transient_smooth`, write directly
- for `df_cropped_spinup_smooth`, calculate typical, cyclic for nyears, then write

In [ ]:
def write_lai(frame, path):
    out = frame.copy(); out['time [s]'] = out['time [d]'] * 86400
    with h5.File(path, 'w') as hdf:
        for key in out: hdf.create_dataset(key, data=out[key].to_numpy())
outputs['modis_prefire_filename_site'] = str(forcing_prefire_dir / f'{site_name}_MODIS_LAI.h5'); write_lai(df_cropped_transient_smooth, outputs['modis_prefire_filename_site'])
if postfire_dates:
    outputs['modis_postfire_filename_site'] = str(forcing_postfire_dir / f'{site_name}_MODIS_LAI.h5'); write_lai(df_cropped_postfire_smooth, outputs['modis_postfire_filename_site'])

phase_plots = [('Prefire transient', prefire_label, df_cropped_transient_smooth)]
if postfire_dates:
    phase_plots.append(('Postfire transient', postfire_label, df_cropped_postfire_smooth))
fig, axes = plt.subplots(1, len(phase_plots), figsize=(7 * len(phase_plots), 4), squeeze=False)
for axis, (phase_name, phase_label_text, frame) in zip(axes[0], phase_plots):
    lai_columns = [key for key in frame.columns if key not in ['time [d]', 'time [s]']]
    axis.plot(frame['time [d]'], frame[lai_columns])
    axis.set_title(f'{phase_name} MODIS LAI (smoothed)\n{phase_label_text}')
    axis.set_xlabel('Phase day')
    axis.set_ylabel('LAI [-]')
    axis.grid(True, alpha=0.3)
    axis.legend(lai_columns, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Compute one 365-day typical water year from the five Oct.-to-Sep. source years.
df = df_cropped_spinup_smooth
if len(df) % 365 != 0:
    raise ValueError(f'Spinup MODIS record count must be divisible by 365, found {len(df)}')
nyears = len(df) // 365
if nyears != len(spinup_dates) // 365:
    raise ValueError(f'Expected {len(spinup_dates) // 365} spinup water years, found {nyears}')

df_yr = [df.iloc[year * 365:(year + 1) * 365].reset_index(drop=True) for year in range(nyears)]
df_avg = pd.DataFrame({
    key: np.mean([yr[key].to_numpy() for yr in df_yr], axis=0)
    for key in df.columns if not key.startswith('time')
})
df_avg['time [d]'] = np.arange(365)

print(f'Computed one 365-day typical water year from {nyears} spinup water years')
df_avg.iloc[:, :-1].plot(figsize=(4,4))
plt.show()
df_avg


In [ ]:
nyears = nyears_cyclic_spinup # to match typical DayMet for cyclic spin-up

# replicate nyears times to make nyears years (remem)
# tile all data to repeat n_year times
df_repeat = pd.DataFrame()
for key in df_avg:
    if not key.startswith('time'):
        df_repeat[key] = np.tile(df_avg[key].array, nyears)
        assert(len(df_repeat) == nyears*365)

# time is simply daily data
df_repeat['time [d]'] = np.arange(0., nyears * 365., 1.)
df_repeat['time [s]'] = 86400*df_repeat['time [d]']

# plot
df_repeat.iloc[:,:-2].plot(figsize=(8,4))
# plot this and make sure it looks right
# fig = plt.figure()
# axs = fig.subplots(3,1)
# plot(df_repeat, '-', axs)
# plt.tight_layout()
plt.show()

df_repeat

In [ ]:
outputs['modis_spinup_filename_site_smoothed'] = str(forcing_spinup_dir / f'{site_name}_MODIS_LAI_cyclic{nyears_cyclic_spinup}y.h5')
if 'time [d]' in df_repeat: df_repeat = df_repeat.drop(columns=['time [d]'])
with h5.File(outputs['modis_spinup_filename_site_smoothed'], 'w') as hdf:
    for key in df_repeat: hdf.create_dataset(key, data=df_repeat[key].to_numpy())


# Merged MODIS-LAI for ats-pflotran transient
- mainly for restart use

In [ ]:
outputs['modis_full_timeline_filename_site'] = str(forcing_full_dir / f'{site_name}_MODIS_LAI.h5')
phases = [('cyclic spinup', df_repeat, nyears_cyclic_spinup * 365), ('prefire transient', df_cropped_transient_smooth, len(prefire_dates))]
if postfire_dates: phases.append(('postfire transient', df_cropped_postfire_smooth, len(postfire_dates)))
for label, frame, expected in phases:
    if len(frame) != expected: raise ValueError(f'{label}: expected {expected}, found {len(frame)}')
total = sum(len(frame) for _, frame, _ in phases); times = np.arange(total) * 86400
print('\nMODIS LAI forcing summary'); print(f'  spinup source period: {spinup_label} ({len(spinup_dates)} days)'); print(f'  cyclic ATS spinup: {nyears_cyclic_spinup} years ({len(df_repeat)} days)'); print(f'  prefire transient: {prefire_label} ({len(prefire_dates)} days)'); print(f'  postfire transient: {postfire_label} ({len(postfire_dates)} days)' if postfire_dates else '  postfire transient: not configured'); print(f'  full timeline: {total} days; time = {times[0]} to {times[-1]} s')
with h5.File(outputs['modis_full_timeline_filename_site'], 'w') as hdf:
    hdf.create_dataset('time [s]', data=times)
    for key in [k for k in df_repeat if k not in ['time [s]', 'time [d]']]:
        values = np.concatenate([frame[key].to_numpy() for _, frame, _ in phases])
        if len(values) != total: raise ValueError(f'MODIS mismatch: {key}')
        hdf.create_dataset(key, data=values)
print(f'Wrote canonical MODIS forcing: {outputs["modis_full_timeline_filename_site"]}')


In [ ]:
outputs